# BSM implied volatility 구현 설명

이 문서는 시장 옵션가격에서 BSM implied volatility(IV)를 역산하도록 구현한 코드의 **전체 구조, 핵심 알고리즘, 검증 순서, 결과와 예외, 테스트 및 SPX Notebook 연결 방식**을 설명한다. 소스 전체를 그대로 복사하기보다, 코드를 읽을 때 먼저 이해해야 하는 흐름과 핵심 분기만 발췌한다.

관련 파일은 다음과 같다.

- IV solver: [implied_volatility.py](../src/option_pricing_volatility/volatility/implied_volatility.py)
- 공개 package export: [volatility/__init__.py](../src/option_pricing_volatility/volatility/__init__.py)
- 재사용하는 BSM pricer: [bsm.py](../src/option_pricing_volatility/models/bsm.py)
- 단위 테스트: [test_implied_volatility.py](../tests/test_implied_volatility.py)
- 분석 Notebook: [05_01_implied_volatility_calculation.ipynb](../notebooks/05_implied_volatility/05_01_implied_volatility_calculation.ipynb)
- 금융·수치 계약: [model_contracts.md](model_contracts.md)
- SPX 데이터 계약: [data_contracts.md](data_contracts.md)

핵심 설계 원칙은 하나다. **BSM 가격 공식은 다시 구현하지 않고 기존 `bsm_price()`를 반복 호출하면서 시장가격을 재현하는 변동성을 찾는다.**

## 1. 전체 구조와 호출 흐름

<pre>
사용자 / 분석 Notebook
        |
        | S, K, T, r, q, option_type, market_price
        v
volatility/implied_volatility.py : implied_volatility()
        |
        |-- scalar 입력과 solver 설정 검증
        |-- 할인된 유럽형 무차익 가격범위 검증
        |-- bsm_price(sigma=0) 경계 확인
        |-- bsm_price(lower), bsm_price(upper)로 root 존재 확인
        |-- 이분탐색으로 volatility bracket 반복 축소
        v
ImpliedVolatilityResult
  volatility | repricing_error | iterations | converged
        |
        +--> tests/test_implied_volatility.py가 금융·수치 계약 검증
        |
        +--> SPX Notebook이 성공값 또는 행별 실패 사유를 tidy table에 저장
</pre>

각 파일의 책임은 다음처럼 분리되어 있다.

| 파일 | 책임 | 포함하지 않는 것 |
|---|---|---|
| `models/bsm.py` | 주어진 변동성의 유럽형 call/put 가격 계산 | root finding, 시장 DataFrame |
| `volatility/implied_volatility.py` | 단일 시장가격의 IV 역산과 진단값 반환 | SPX schema, 반복 행 처리, plotting |
| `tests/test_implied_volatility.py` | round-trip, 무차익, bracket, 수렴·실패 계약 보호 | live market data |
| 분석 Notebook | synthetic 검증, SPX 행별 orchestration, `iv_skew_df` 구성 | BSM 공식 또는 solver 재구현 |

따라서 solver는 SPX에 종속되지 않는다. 다른 기초자산도 동일한 단위와 BSM 가정을 만족하는 scalar 입력으로 호출할 수 있다.

## 2. 역산 문제와 이분탐색 원리

시장가격을 $V_{market}$, 기존 BSM 함수가 계산한 가격을 $V_{BSM}(\sigma)$라 하면 solver가 푸는 식은 다음과 같다.

$$
f(\sigma)=V_{BSM}(\sigma)-V_{market}=0
$$

일반적인 유럽형 call과 put의 BSM 가격은 다른 입력이 고정되어 있을 때 변동성에 대해 단조 증가한다. 따라서 bracket 양 끝에서

$$
f(\sigma_{lower}) \le 0 \le f(\sigma_{upper})
$$

가 성립하면 중간 변동성의 가격을 확인해 root가 있는 절반 구간만 남길 수 있다. 이 과정을 반복하면 매번 bracket 폭이 절반이 된다.

한 반복의 판단은 단순하다.

- `midpoint_price > market_price`: 변동성이 너무 크므로 upper를 midpoint로 내린다.
- `midpoint_price < market_price`: 변동성이 너무 작으므로 lower를 midpoint로 올린다.
- 가격 오차가 충분히 작거나 남은 변동성 구간이 충분히 좁으면 종료한다.

Newton–Raphson처럼 vega와 초기 추정값을 요구하지 않으며, root가 bracket 안에 있다는 사전 검증을 통과한 뒤에는 안정적으로 구간을 축소한다는 것이 이 구현의 목적이다.

## 3. 공개 API와 결과 객체

공개 함수의 signature는 다음과 같다.

<pre><code class="language-python">def implied_volatility(
    spot: float,
    strike: float,
    maturity: float,
    rate: float,
    market_price: float,
    option_type: str,
    dividend_yield: float = 0.0,
    *,
    volatility_lower: float = 1e-8,
    volatility_upper: float = 5.0,
    price_tolerance: float = 1e-8,
    volatility_tolerance: float = 1e-12,
    max_iterations: int = 200,
) -&gt; ImpliedVolatilityResult:</code></pre>

| 인수 | 의미와 convention |
|---|---|
| `spot`, `strike` | 같은 가격 단위의 양수 |
| `maturity` | ACT/365F 연 단위, 반드시 `T > 0` |
| `rate`, `dividend_yield` | 연속복리 연율 소수 |
| `market_price` | IV가 재현해야 할 옵션가격 |
| `option_type` | 정확히 `call` 또는 `put` |
| `volatility_lower`, `volatility_upper` | 연율 소수 단위의 탐색 bracket |
| `price_tolerance` | `abs(BSM_price - market_price)` 종료 기준 |
| `volatility_tolerance` | 남은 volatility bracket 폭 종료 기준 |
| `max_iterations` | 반복 횟수 상한 |

여러 진단값이 함께 필요하므로 결과는 immutable dataclass로 반환한다.

<pre><code class="language-python">@dataclass(frozen=True, slots=True)
class ImpliedVolatilityResult:
    volatility: float
    repricing_error: float
    iterations: int
    converged: bool</code></pre>

`repricing_error`는 절대값이 아니라 `BSM_price - market_price`의 signed error다. 양수이면 복원한 IV의 BSM 가격이 시장가격보다 높고, 음수이면 낮다. 정상 반환은 항상 `converged=True`이며, 실패는 `converged=False` 객체가 아니라 예외로 드러난다.

## 4. 입력 검증과 형변환

solver는 root finding 전에 잘못된 입력을 먼저 차단한다. 공통 수치 입력은 dictionary로 모아 같은 규칙을 적용한다.

<pre><code class="language-python">for name, value in numeric_inputs.items():
    if (
        isinstance(value, bool)
        or not isinstance(value, Real)
        or not math.isfinite(value)
    ):
        raise ValueError(f"{name} must be a finite real number")</code></pre>

`bool`은 Python에서 정수의 하위 타입이므로 별도로 먼저 거부한다. 이 검사가 없으면 `True`가 가격 1 또는 반복 횟수 1처럼 사용될 수 있다. `NaN`, 양·음의 무한대, 문자열도 이 단계에서 거부된다.

그 다음 금융 도메인과 solver 설정을 검사한다.

- `spot > 0`, `strike > 0`, `maturity > 0`
- `option_type in {"call", "put"}`
- `volatility_lower >= 0`
- `volatility_upper > volatility_lower`
- 두 tolerance는 모두 양수
- `max_iterations`는 bool이 아닌 1 이상의 정수

BSM 가격 함수는 만기 시점 `T=0` 가격을 계산할 수 있지만, 그때 가격은 변동성과 무관하므로 IV가 하나로 식별되지 않는다. 따라서 pricer와 달리 IV solver는 `maturity=0`을 거부한다.

검증을 통과한 값은 내장 `float`와 `int`로 변환한다. 이후 할인값이나 극단적인 bracket endpoint 계산이 overflow하면 임의로 값을 clipping하지 않고 설명 가능한 `ValueError`로 바꾼다.

## 5. 할인된 무차익 범위 검증

수치적으로 root를 찾기 전에 시장가격이 유럽형 옵션의 이론적 가격범위 안에 있는지 확인한다. 먼저

$$
S_d=Se^{-qT}, \qquad K_d=Ke^{-rT}
$$

를 계산한다. call과 put의 범위는 다음과 같다.

$$
\max(0,S_d-K_d) \le C \le S_d
$$

$$
\max(0,K_d-S_d) \le P \le K_d
$$

코드는 option type에 따라 lower와 upper를 만든 뒤 시장가격을 비교한다.

<pre><code class="language-python">if option_type == "call":
    arbitrage_lower = max(0.0, discounted_spot - discounted_strike)
    arbitrage_upper = discounted_spot
else:
    arbitrage_lower = max(0.0, discounted_strike - discounted_spot)
    arbitrage_upper = discounted_strike

if market_price &lt; arbitrage_lower or market_price &gt; arbitrage_upper:
    raise ValueError(...)</code></pre>

이 범위는 입력 유효성 계약이므로 `price_tolerance`만큼 넓히지 않는다. 예를 들어 lower bound보다 아주 조금 낮은 가격도 `sigma=0` 가격과 가깝다는 이유로 IV 0으로 바꾸지 않는다. 이는 잘못된 시장가격을 clipping해 성공처럼 보이게 만드는 것을 방지한다.

무차익 범위와 volatility bracket 범위는 서로 다르다. 무차익 범위 안의 가격이라도 유한한 최대 변동성 `5.0`으로 만들 수 없다면 현재 solver bracket에는 root가 없는 것이다.

## 6. `sigma=0` 경계와 bracket root 확인

기본 양의 bracket은 `[1e-8, 5.0]`이지만 IV 0은 별도의 경제적 경계로 지원한다. 무차익 검증을 통과한 뒤 기존 BSM 함수에 정확히 `volatility=0.0`을 전달한다.

<pre><code class="language-python">zero_price = bsm_price(
    spot, strike, maturity, rate, 0.0, option_type, dividend_yield
)
zero_error = zero_price - market_price
if abs(zero_error) &lt;= price_tolerance:
    return ImpliedVolatilityResult(
        volatility=0.0,
        repricing_error=zero_error,
        iterations=0,
        converged=True,
    )</code></pre>

여기서 작은 양의 변동성으로 대체하지 않는 이유는 `bsm_price()` 자체가 `sigma=0`의 할인된 결정론적 payoff를 명시적으로 구현하고 있기 때문이다.

0-IV가 아니면 두 bracket endpoint를 BSM으로 가격화한다.

<pre><code class="language-python">lower_price = bsm_price(..., volatility_lower, ...)
upper_price = bsm_price(..., volatility_upper, ...)

if market_price &lt; lower_price or market_price &gt; upper_price:
    raise ValueError("market_price is not attainable within volatility bracket ...")</code></pre>

이 검사는 이분탐색의 전제인 root bracket을 보장한다. 시장가격이 endpoint 가격범위 밖이면 endpoint를 답처럼 반환하지 않고 실패한다. 반대로 시장가격이 lower 또는 upper endpoint 가격과 `price_tolerance` 안에서 일치하면 반복 없이 그 endpoint를 정상 결과로 반환한다.

## 7. 이분탐색 반복문의 핵심

endpoint 검증 후 실제 반복은 다음 구조다.

<pre><code class="language-python">for iteration in range(1, max_iterations + 1):
    midpoint = 0.5 * (lower + upper)
    midpoint_price = bsm_price(
        spot, strike, maturity, rate, midpoint, option_type, dividend_yield
    )
    latest_error = midpoint_price - market_price

    if abs(latest_error) &lt;= price_tolerance:
        return ImpliedVolatilityResult(...)

    if latest_error &gt; 0.0:
        upper = midpoint
    else:
        lower = midpoint

    if upper - lower &lt;= volatility_tolerance:
        estimate = 0.5 * (lower + upper)
        estimate_price = bsm_price(..., estimate, ...)
        return ImpliedVolatilityResult(...)</code></pre>

코드 흐름을 줄 단위로 해석하면 다음과 같다.

1. 현재 lower와 upper의 정확한 중간 변동성을 계산한다.
2. 기존 `bsm_price()`로 그 변동성의 옵션가격을 계산한다.
3. signed error가 가격 tolerance 안이면 midpoint를 IV로 반환한다.
4. error가 양수이면 midpoint 변동성이 너무 크므로 upper를 내린다.
5. error가 음수이면 midpoint 변동성이 너무 작으므로 lower를 올린다.
6. 갱신된 bracket 폭이 volatility tolerance 안이면 새 bracket의 중앙값을 최종 추정값으로 삼고 한 번 더 재가격해 일관된 error를 저장한다.

각 반복에서 bracket 폭이 절반이므로 시간복잡도는 요구한 변동성 정밀도에 대해 $O(\log((\sigma_{upper}-\sigma_{lower})/\epsilon_\sigma))$이고, 추가 메모리는 $O(1)$이다. BSM 가격 공식을 Notebook이나 solver 내부에 복사하지 않았기 때문에 BSM convention이 한 곳에서 유지된다.

## 8. 종료 조건, 결과 해석, 실패 동작

정상 종료 조건은 OR 관계다.

- `abs(repricing_error) <= price_tolerance`
- `upper - lower <= volatility_tolerance`

따라서 `converged=True`는 둘 중 하나를 만족했다는 의미다. bracket 폭 기준으로 먼저 종료된 경우 `abs(repricing_error)`가 price tolerance보다 클 수 있으므로, 가격 오차가 특히 중요한 호출자는 반환된 `repricing_error`도 직접 확인해야 한다.

예외는 실패 원인을 구분한다.

| 상황 | 예외 | 의미 |
|---|---|---|
| 비유한값, 잘못된 도메인 또는 solver 설정 | `ValueError` | 입력 계약 위반 |
| 할인된 무차익 범위 밖 시장가격 | `ValueError` | 유효한 유럽형 가격이 아님 |
| bracket endpoint 가격범위 밖 시장가격 | `ValueError` | 설정한 bracket 안에 root가 없음 |
| endpoint 계산 overflow | `ValueError` | 계산 가능한 bracket이 아님 |
| `max_iterations`까지 두 종료 기준 미충족 | `RuntimeError` | 명시적 nonconvergence |

마지막 경우의 메시지에는 마지막 절대 가격오차와 bracket 폭이 포함된다. 재사용 solver는 실패할 때 `NaN`을 반환하지 않는다. 여러 시장 행을 처리하는 Notebook 계층이 예외를 잡아 `NaN`과 `iv_failure_reason`으로 변환하며, solver 자체는 성공과 실패를 모호하게 섞지 않는다.

## 9. 단위 테스트가 보호하는 동작

`tests/test_implied_volatility.py`는 live data 없이 BSM으로 만든 synthetic 가격과 금융 불변조건을 사용한다. 핵심 테스트는 다음과 같다.

| 테스트 범주 | 구성 | 보호하는 동작 |
|---|---|---|
| call/put round-trip | 알려진 `sigma=0.24`로 시장가격 생성 후 IV 복원 | 두 option type의 역산 정확성 |
| repricing | 복원한 IV를 `bsm_price()`에 다시 입력 | signed error와 가격 tolerance |
| zero volatility | `bsm_price(..., volatility=0)`을 target으로 사용 | 정확한 `0.0`, 반복 0회 반환 |
| 무차익 위반 | call/put upper bound보다 큰 가격 | root finding 전 거부 |
| bracket root 부재 | 실제 IV보다 낮은 custom upper 사용 | endpoint를 답으로 반환하지 않음 |
| strict bounds | lower bound보다 tolerance 이내로 작은 가격 | tolerance로 무차익 범위를 확장하지 않음 |
| invalid inputs | 가격·만기·bracket·tolerance·반복 설정 변경 | `ValueError` 계약 |
| 반복 소진 | tolerance를 매우 작게 하고 1회만 허용 | `RuntimeError` 계약 |
| bracket-width 수렴 | 가격 tolerance보다 폭 기준이 먼저 충족되게 설정 | 두 번째 정상 종료 경로 |
| 극단 bracket | 매우 큰 finite upper | raw overflow 대신 명시적 `ValueError` |

round-trip 테스트의 구조는 다음과 같다.

<pre><code class="language-python">market_price = bsm_price(..., volatility=original_volatility, ...)
result = implied_volatility(..., market_price=market_price, ...)
assert result.volatility == pytest.approx(original_volatility, abs=1e-9)</code></pre>

구현 완료 시 IV 테스트 26개와 저장소 전체 테스트 88개가 통과했다. 테스트는 현재 시장상황이나 API에 의존하지 않으므로 반복 실행해도 결정적이다.

## 10. 분석 Notebook과 SPX 행별 계산 연결

분석 Notebook은 solver를 재구현하지 않고 다음 두 역할만 수행한다.

### 10.1 Synthetic 확인

call과 put에 대해 알려진 변동성으로 BSM 가격을 만든 뒤 IV를 복원하여 다음 tidy table을 만든다.

```text
option_type | original_volatility | recovered_implied_volatility
            | volatility_error    | repricing_error
```

이는 패키지 테스트와 같은 round-trip을 사람이 읽을 수 있는 표로 보여준다.

### 10.2 고정 SPX snapshot 처리

processed SPX의 시장가격은 데이터 계약에 따라 `target_price = mid`이고, 표시용 `market_price`도 같은 값을 사용한다. `last`나 provider IV는 대체값으로 사용하지 않는다. 각 행은 `implied_volatility()`을 호출하고 다음처럼 결과를 기록한다.

- 성공: `implied_volatility`, `repricing_error`, 반복 횟수와 success status
- 실패: IV와 repricing error는 `NaN`, status는 failed, 예외 또는 누락 입력을 `iv_failure_reason`에 기록

실패 행을 삭제하지 않으므로 어떤 계약이 IV를 거부했는지 추적할 수 있다. 최종 `iv_skew_df`에는 strike, option type, market price, 가능한 moneyness 열, IV, repricing error와 failure reason이 남는다.

현재 local SPX processed snapshot에는 출처가 기록된 `risk_free_rate`, `dividend_yield`, `forward`가 없다. Notebook은 임의의 r/q나 forward를 만들지 않고 424행 모두를 `MISSING_MODEL_INPUT:risk_free_rate,dividend_yield`로 보존한다. 향후 출처 있는 r/q 열이 입력에 추가되면 같은 행별 코드가 실제 IV를 계산한다. 기존 `forward`가 있을 때만 `log(strike / forward)`를 계산하고, 없으면 strike 기준으로 정렬한다.

## 11. 코드를 읽는 순서와 현재 범위

처음 코드를 확인할 때는 다음 순서가 가장 짧다.

1. `bsm_price()`의 인수 순서와 `sigma=0` 동작을 확인한다.
2. `ImpliedVolatilityResult`로 정상 반환값의 구조를 확인한다.
3. `implied_volatility()` 상단의 일반 입력 검증과 무차익 범위를 읽는다.
4. 0-IV 및 bracket endpoint 검증을 읽어 반복문 진입 조건을 확인한다.
5. 반복문에서 signed error에 따라 어느 endpoint가 움직이는지 확인한다.
6. 테스트에서 정상 round-trip과 각 실패가 어떻게 재현되는지 확인한다.
7. 마지막으로 분석 Notebook에서 scalar API를 시장 행에 연결하는 방법을 본다.

현재 구현 범위는 scalar European call/put IV와 plotting-ready SPX 결과 준비까지다. 다음 항목은 의도적으로 포함하지 않는다.

- Newton–Raphson, Brent 등 다른 root solver
- 배열 전용 vectorized solver 또는 병렬화
- volatility surface, smoothing, interpolation
- bid-IV/ask-IV band, Greeks 또는 low-vega 품질 확장
- ATM volatility 선택과 최종 그래프

이 범위를 유지하면 재사용 함수는 작은 scalar 수치 모듈로 남고, 데이터 선택과 실패 사유 보존은 Notebook/분석 계층에서 별도로 관리할 수 있다.